# PCU-READOUT-LOCALIZATION-001 — Minimal single-layer readout judgment

Engineering-only observational diagnostic. It replays the already published L7/K64 hybrid training without changing any training variable, requires exact reproduction of the published hybrid metrics, then measures gold-prefix target-token ranks and forced-prefix greedy suffix recovery.

Interpretation: if later target tokens become top-1 under correct prefixes, the problem is early-token/trajectory readout; if even gold-prefix later tokens remain poor, that is direct evidence that the single-layer mutation does not adequately control native token readout. Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
HYBRID = REPO / 'artifacts/research/pcu-hybrid-objective-001/engineering/26090501-l7-k64-rank-plus-ce025'
OBJECTIVE = REPO / 'artifacts/research/pcu-objective-alignment-001/engineering/26090501-l7-k64-ranking'
OUT = REPO / 'artifacts/research/pcu-readout-localization-001/engineering/26090501-l7-k64-hybrid-readout'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'hybrid_core_blob': run(['git', 'rev-parse', 'HEAD:src/minicells/pcu_kill_001/hybrid_objective.py'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'transformers': transformers.__version__,
}, indent=2))
assert run(['git', 'rev-parse', 'HEAD:src/minicells/pcu_kill_001/hybrid_objective.py'], capture=True) == '851c77cdd283def0698ebe721ea8bf216f5ed556'


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
print(json.dumps({'formal_seed_states': formal_states()}, indent=2))


In [ ]:
required_hybrid = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
assert not [name for name in required_hybrid if not (HYBRID / name).is_file()]
hybrid_decision = json.loads((HYBRID / 'DECISION.json').read_text())
assert hybrid_decision['status'] == 'HYBRID_OBJECTIVE_PRESERVES_ASSOCIATION_GENERATION_UNRESOLVED'
assert abs(hybrid_decision['ranking_train_accuracy'] - 1.0) < 1e-12
assert abs(hybrid_decision['ranking_eval_accuracy'] - 0.8359375) < 1e-12
assert abs(hybrid_decision['direct_accuracy'] - 0.03125) < 1e-12
assert abs(hybrid_decision['ce_weight'] - 0.25) < 1e-12
remote_path = 'artifacts/research/pcu-hybrid-objective-001/engineering/26090501-l7-k64-rank-plus-ce025/DECISION.json'
assert subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote_path}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
print(json.dumps({'hybrid_prerequisite': hybrid_decision['status'], 'published': True}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU readout-localization test/compile gate: PASS')


In [ ]:
if OUT.exists():
    existing = sorted(p.name for p in OUT.glob('*.json'))
    assert not existing, f'Readout-localization output already exists; inspect before rerun: {existing}'
run([
    sys.executable, 'scripts/research/run_pcu_readout_localization_001.py',
    '--seed', '26090501',
    '--device', 'cuda:0',
    '--hybrid-baseline', HYBRID,
    '--objective-baseline', OBJECTIVE,
    '--out', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert decision['valid_run'] is True
assert decision['formal_execution_not_started'] is True
assert decision['training_changed'] is False
assert decision['hybrid_reproduction_exact'] is True
assert decision['selected_cells_exact_hybrid_match'] is True
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'first_token_top1_accuracy': decision['first_token_top1_accuracy'],
    'later_token_top1_accuracy': decision['later_token_top1_accuracy'],
    'all_token_top1_accuracy': decision['all_token_top1_accuracy'],
    'sequence_all_tokens_top1_accuracy': decision['sequence_all_tokens_top1_accuracy'],
    'first_token_mean_target_rank': decision['first_token_mean_target_rank'],
    'later_token_mean_target_rank': decision['later_token_mean_target_rank'],
    'force0_suffix_exact_accuracy': decision['force0_suffix_exact_accuracy'],
    'force1_suffix_exact_accuracy': decision['force1_suffix_exact_accuracy'],
    'force2_suffix_exact_accuracy': decision['force2_suffix_exact_accuracy'],
    'minimal_forced_tokens_reaching_floor': decision['minimal_forced_tokens_reaching_floor'],
    'hybrid_reproduction': result['hybrid_reproduction'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_readout_localization_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
